In [ ]:
!pip install -q -U transformers accelerate mistral-common

In [ ]:
!pip install -q --force-reinstall --no-deps torchvision==0.29.0

In [2]:
!pip install -q -U laya

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.8/90.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 13.8 MB/s eta 0:00:00


In [3]:
from laya import Router

laya_router = Router(
    device="cuda",
    default="english"
)

In [4]:
import json
import torch

In [5]:
import torch
import transformers

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Torch: 2.10.0+cu128
Transformers: 5.0.0
CUDA: True
GPU: Tesla T4


In [8]:
import transformers

print("Transformers:", transformers.__version__)

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

# ---


Transformers: 5.0.0


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [9]:
from laya import Router

laya_router = Router(
    device="cuda",
    default="english"
)

In [10]:
# Testing the raw query (without ADHD brain):
prompt = "Ways to grow my instagram followers from 1K to 10K in a week?"

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.7,
    do_sample=True
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

<think>
Okay, the user wants to grow their Instagram followers from 1K to 10K in a week. That's a big jump, and I need to figure out the best strategies. Let me start by recalling what's effective for growing Instagram followers.

First, I remember that Instagram is a visual platform, so content quality is key. Maybe they need to focus on creating high-quality, engaging posts. But how to do that quickly? Maybe they need to post consistently, but even that might take time. Wait, but they have a week, so maybe they can optimize their content.

Engagement is crucial. If they can get likes, comments, and shares, that can help their posts go viral. But how to boost engagement in a week? Maybe using hashtags effectively. But they need to research the right hashtags for their niche. But if they're not sure, maybe they can use popular hashtags first.

Influencer collaborations could be a way. If they partner with influencers in their niche, they can get more followers. But how to find those in

In [11]:
state = {
    "problem": "",

    "ideas": [],
    "connections": [],
    "perspectives": [],
    "attention_perturbations": [],

    "cross_connections": [],
    "selected_candidates": [],

    "final_ideas": [],

    "missing_directions": [],

    "active_threads": [],
    "discarded_ideas": [],
    "active_memory": [],

    "round": 0
}

In [12]:
state["problem"] = "Ways to grow my instagram followers from 1K to 10K in a week?"

In [13]:
divergence_modes = [
    "direct",
    "adjacent",
    "unrelated",
    "analogical",
    "first_principles"
]

divergence_instructions = {

    "direct":
        "Stay within the problem's domain. Generate ideas using knowledge, strategies, and mechanisms directly related to the problem.",

    "adjacent":
        "Look at closely related fields or neighboring domains. Borrow useful concepts, strategies, or mechanisms from areas that are connected to the problem.",

    "unrelated":
        "Look outside the problem's normal domain. Borrow useful concepts, mechanisms, strategies, or structures from unrelated fields.",

    "analogical":
        "Find systems, situations, or problems that are structurally similar to the current problem. Use those analogies to generate new ideas.",

    "first_principles":
        "Break the problem down into its fundamental assumptions and underlying principles. Generate ideas from those basic principles rather than relying on conventional solutions."
}


def diverge(state):

    all_ideas = []

    for mode in divergence_modes:
        prompt = f"""
        Problem:
        {state["problem"]}
        
        Previous missing directions (if it's running for second round):
        {state["missing_directions"]}
        
        Divergence mode:
        {mode}
        
        Instruction:
        {divergence_instructions[mode]}
        
        Generate exactly 4 ideas using this divergence mode.
        
        The ideas must be meaningfully different from each other.
        
        If previous missing directions are provided, use them
        to explore those areas while still following the
        current divergence mode.
        
        Do not simply rephrase existing ideas.
        
        Return ONLY valid JSON in this format:
        
        {{
            "ideas": [
                "idea 1",
                "idea 2"
            ]
        }}
    """

        messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )

        inputs = tokenizer(
            text,
            return_tensors="pt"
        ).to(model.device)

        with torch.inference_mode():

            outputs = model.generate(
                **inputs,
                max_new_tokens=300,
                temperature=0.8,
                do_sample=True
            )

        response = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        del inputs
        del outputs
        torch.cuda.empty_cache()

        result = json.loads(response)

        all_ideas.extend(result["ideas"])

    state["ideas"].extend(all_ideas)

    return all_ideas

In [14]:
association_types = [
    "concepts",
    "mechanisms",
    "analogies",
    "principles",
    "applications"
]

association_instructions = {

    "concepts":
        "Identify concepts that are meaningfully associated with the given idea. Look for both obvious and less obvious conceptual relationships.",

    "mechanisms":
        "Identify mechanisms that could make the given idea work. Break the idea into underlying processes, causes, or mechanisms.",

    "analogies":
        "Find systems, objects, situations, or fields that work in a similar way to the given idea. Use those similarities to generate new possibilities.",

    "principles":
        "Identify deeper principles or fundamental ideas underlying the given idea. Use those principles to discover new possibilities.",

    "applications":
        "Explore different domains, situations, or use cases where the given idea or its underlying mechanism could be applied."
}


def associative_search(state):

    all_connections = []

    for association_type in association_types:

        prompt = f"""
            Problem:
            {state["problem"]}
            
            Ideas:
            {state["ideas"]}
            
            Association type:
            {association_type}
            
            Instruction:
            {association_instructions[association_type]}
            
            Generate exactly 3 useful associations using this
            association type.
            
            The associations should add genuinely new information
            or possibilities rather than simply restating the ideas.
            
            Focus specifically on the association type provided.
            
            Return ONLY valid JSON in this format:
            
            {{
                "connections": [
                    "connection 1",
                    "connection 2"
                ]
            }}
        """

        messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )

        inputs = tokenizer(
            text,
            return_tensors="pt"
        ).to(model.device)

        # Cuda ops starts
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=300,
                temperature=0.8,
                do_sample=True
            )

        response = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        del inputs
        del outputs
        torch.cuda.empty_cache()
        # cuda ops ends
        
        result = json.loads(response)

        all_connections.extend(result["connections"])

    state["connections"].extend(all_connections)

    return all_connections

In [15]:
perspectives = [
    "entrepreneur",
    "engineer",
    "investor",
    "scientist",
    "artist",
    "customer",
    "competitor",
    "outsider",
    
    "Elon_musk",
    "Adolf_Hitler",
    "Common_labourer_in_India",
    "Richard_feynman",
    "Gautama_Buddha"
]


def perspective_switch(state):

    all_perspectives = []

    for perspective in perspectives:

        prompt = f"""
            Problem:
            {state["problem"]}
            
            Current ideas:
            {state["ideas"]}

            Perspective:
            {perspective}
            
            Analyze the problem from this perspective.
            
            Do not simply describe what this person or role
            would think.
            
            Instead, use the perspective as a reasoning lens.
            Identify assumptions, priorities, constraints,
            opportunities, or solution approaches that might
            become visible from this perspective.
            
            Generate exactly 1 genuinely different perspective-based insights.
            
            The insights should add new information or
            possibilities rather than simply restating the
            existing ideas.
            
            For historical figures, treat the perspective as
            analysis of their documented worldview, methods,
            or decision-making style. Do not endorse harmful
            or extremist ideology.
            
            Return ONLY valid JSON in this format:
            
            {{
                "perspectives": [
                    "insight"
                ]
            }}
        """

        messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )

        inputs = tokenizer(
            text,
            return_tensors="pt"
        ).to(model.device)

        # Cuda ops starts
        with torch.inference_mode():

            outputs = model.generate(
                **inputs,
                max_new_tokens=300,
                temperature=0.8,
                do_sample=True
            )

        response = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        del inputs
        del outputs
        torch.cuda.empty_cache()
        # cuda ops ends

        result = json.loads(response)

        all_perspectives.extend(
            result["perspectives"]
        )

    state["perspectives"].extend(
        all_perspectives
    )

    return all_perspectives

In [16]:
perturbation_types = [
    "reversed_assumption",
    "removed_constraint",
    "extreme_constraint",
    "changed_objective",
    "scale_change",
    "opposite_solution"
]

perturbation_instructions = {

    "reversed_assumption":
        "Identify an important assumption behind the idea and reverse it. Explore what solution becomes possible when the assumption is false or opposite.",

    "removed_constraint":
        "Remove an important constraint from the idea. Explore what could be done if that constraint did not exist.",

    "extreme_constraint":
        "Make an important constraint much more extreme. Explore how the solution changes under this extreme condition.",

    "changed_objective":
        "Change the objective or goal of the idea. Explore what solution emerges when the system optimizes for a different objective.",

    "scale_change":
        "Change the scale of the idea dramatically. Consider what happens at a much smaller or much larger scale.",

    "opposite_solution":
        "Reverse the core approach of the idea. Explore a solution that works in the opposite direction or uses the opposite strategy."
}

def attention_perturb(state):

    all_perturbations = []

    for perturbation_type in perturbation_types:

        prompt = f"""
            Problem:
            {state["problem"]}
            
            Ideas:
            {state["ideas"]}
            
            Perturbation type:
            {perturbation_type}
            
            Instruction:
            {perturbation_instructions[perturbation_type]}
            
            Generate exactly 3 useful perturbations using this
            perturbation type.
            
            The perturbations should meaningfully change the
            original idea rather than simply rephrasing it.
            
            Focus specifically on the perturbation type provided.
            
            Return ONLY valid JSON in this format:
            
            {{
                "attention_perturbations": [
                    "perturbation 1",
                    "perturbation 2"
                ]
            }}
        """

        messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )

        inputs = tokenizer(
            text,
            return_tensors="pt"
        ).to(model.device)

        # Cuda ops starts
        with torch.inference_mode():

            outputs = model.generate(
                **inputs,
                max_new_tokens=300,
                temperature=0.8,
                do_sample=True
            )

        response = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        del inputs
        del outputs
        torch.cuda.empty_cache()
        # cuda ops ends

        result = json.loads(response)

        all_perturbations.extend(
            result["attention_perturbations"]
        )

    state["attention_perturbations"].extend(
        all_perturbations
    )

    return all_perturbations

In [17]:
# def cross_connect(state):

#     prompt = f"""
#         Problem:
#         {state["problem"]}
        
#         Ideas:
#         {state["ideas"]}
        
#         Associative connections:
#         {state["connections"]}
        
#         Perspectives:
#         {state["perspectives"]}
        
#         Attention perturbations:
#         {state["attention_perturbations"]}
        
#         Find meaningful connections between different elements
#         of the current reasoning.
        
#         Look for:
        
#         - combinations that could create a new solution
#         - unexpected relationships
#         - complementary ideas
#         - ideas that could solve each other's weaknesses
#         - mechanisms from one idea that could improve another
#         - connections between different perspectives or perturbations
        
#         Generate exactly 10 novel cross-connections.
        
#         Each cross-connection should be concise, preferably
#         one sentence.
        
#         Do not simply repeat or rephrase existing ideas.
        
#         Return ONLY valid JSON in exactly this format:
        
#         {{
#             "cross_connections": [
#                 "connection 1",
#                 "connection 2",
#                 "connection 3",
#                 "connection 4",
#                 "connection 5",
#                 "connection 6",
#                 "connection 7",
#                 "connection 8",
#                 "connection 9",
#                 "connection 10"
#             ]
#         }}
#     """

#     messages = [
#         {
#             "role": "user",
#             "content": prompt
#         }
#     ]

#     text = tokenizer.apply_chat_template(
#         messages,
#         tokenize=False,
#         add_generation_prompt=True,
#         enable_thinking=False
#     )

#     inputs = tokenizer(
#         text,
#         return_tensors="pt"
#     ).to(model.device)

#     with torch.inference_mode():

#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=600,
#             temperature=0.7,
#             do_sample=True
#         )

#     response = tokenizer.decode(
#         outputs[0][inputs["input_ids"].shape[1]:],
#         skip_special_tokens=True
#     )

#     del inputs
#     del outputs
#     torch.cuda.empty_cache()

#     result = json.loads(response)

#     state["cross_connections"].extend(
#         result["cross_connections"]
#     )

#     return result["cross_connections"]


# ### V2
# ### V2
# ###
# def cross_connect(state):

#     # Collect everything generated by the independent
#     # exploration branches.

#     candidates = (
#         state["ideas"]
#         + state["connections"]
#         + state["perspectives"]
#         + state["attention_perturbations"]
#     )

#     state["cross_connections"] = candidates

#     print(
#         f"Prepared {len(candidates)} candidates for Laya"
#     )

#     return candidates

In [18]:
import random
import re

def _generate_json(prompt, max_new_tokens=200, temperature=0.9):
    """Run one generation and safely parse JSON. Returns a dict, or {} on failure."""
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    del inputs
    del outputs
    torch.cuda.empty_cache()

    # Strip code fences, then grab the first {...} block
    response = response.replace("```json", "").replace("```", "").strip()
    match = re.search(r"\{.*\}", response, re.DOTALL)
    if not match:
        return {}
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return {}


def cross_connect(state, n_pairs=25):
    """
    ADDITIVE cross-connect.
    Keeps every original candidate and adds new ideas made by combining
    two fragments from DIFFERENT sources (e.g. an idea + a perturbation).
    """

    # 1. Keep the source of each fragment so we can pair different kinds
    sources = {
        "ideas": state["ideas"],
        "connections": state["connections"],
        "perspectives": state["perspectives"],
        "perturbations": state["attention_perturbations"],
    }
    sources = {k: v for k, v in sources.items() if len(v) > 0}

    # 2. The originals: these ALL go to Laya, no matter what
    originals = []
    for items in sources.values():
        originals.extend(items)

    # 3. Build new mixed ideas from random pairs
    new_ideas = []
    seen = set(originals)
    source_names = list(sources.keys())

    attempts = 0
    max_attempts = n_pairs * 3   # some pairs will be rejected, so allow retries

    while len(new_ideas) < n_pairs and attempts < max_attempts:
        attempts += 1

        # Pick two DIFFERENT source types when possible
        if len(source_names) >= 2:
            name_a, name_b = random.sample(source_names, 2)
        else:
            name_a = name_b = source_names[0]

        frag_a = random.choice(sources[name_a])
        frag_b = random.choice(sources[name_b])

        if frag_a == frag_b:
            continue

        prompt = f"""
Problem:
{state["problem"]}

Fragment A:
{frag_a}

Fragment B:
{frag_b}

Task:
Write ONE new, concrete idea for the problem that works ONLY because
it combines both fragments. It must include a specific first action.

Do not just restate A or B.
Do not use the words "leverage", "utilize" or "implement".

If the two fragments have no real connection, return {{"idea": null}}.

Return ONLY valid JSON in exactly this format:

{{"idea": "your idea here"}}
"""

        result = _generate_json(prompt)
        idea = result.get("idea")

        # Skip failures, nulls, and duplicates
        if not idea or not isinstance(idea, str):
            continue
        idea = idea.strip()
        if len(idea) < 20 or idea in seen:
            continue

        seen.add(idea)
        new_ideas.append(idea)

    # 4. ADDITIVE: originals + new mixed ideas
    candidates = originals + new_ideas
    state["cross_connections"] = candidates

    print(
        f"Cross-connect: {len(originals)} originals + "
        f"{len(new_ideas)} new = {len(candidates)} candidates for Laya"
    )

    return candidates

In [25]:
def laya_select(state):

    candidates = state["cross_connections"]

    selected_candidates = []
    decisions = []
    
    print("Laya decision making:\n")

    for i, candidate in enumerate(candidates):

        laya_state = {
            "problem": state["problem"],
            "candidate": candidate
        }

        questions = {
            "keep": {
                "type": "noul",

                "instructions": (
                    "Should this candidate be kept as a promising "
                    "solution to the original problem?"
                ),

                "criteria": {
                    "true": (
                        "Keep the candidate if it is relevant to "
                        "the problem, potentially useful, sufficiently "
                        "novel, and reasonably feasible."
                    ),

                    "false": (
                        "Reject the candidate if it is irrelevant, "
                        "redundant, impractical, or does not provide "
                        "a meaningful solution direction."
                    )
                },

                "labels": {
                    "true": "A",
                    "false": "B"
                }
            }
        }

        result = laya_router.predict(
            laya_state,
            questions,
            model="english"
        )

        probability = result["answers"]["keep"]["noul"]

        keep = probability >= 0.5

        print(
            f"Candidate {i+1}: "
            f"P(true)={probability:.4f} | "
            f"keep={keep}"
        )

        decisions.append({
            "candidate": candidate,
            "probability": probability,
            "keep": keep
        })

        if keep:
            selected_candidates.append(candidate)

        torch.cuda.empty_cache()

    state["selected_candidates"] = selected_candidates

    torch.cuda.empty_cache()

    print(
        f"\nLaya selected "
        f"{len(selected_candidates)} / {len(candidates)} candidates"
    )

    return decisions

In [20]:
def evaluate_coverage(state):

    selected = state["selected_candidates"]

    prompt = f"""
            Problem:
            {state["problem"]}
            
            Selected candidates:
            {selected}
            
            Your task is to evaluate the SOLUTION-SPACE COVERAGE
            of these candidates.
            
            Do not judge which candidate is best.
            Do not generate new solutions.
            
            Analyze the underlying strategy or mechanism of
            each candidate.
            
            First identify the distinct solution directions
            represented by the candidates.
            
            Then identify important solution directions that
            are NOT represented.
            
            Two candidates should be considered part of the
            same direction if they rely on essentially the same
            underlying mechanism, even if their wording differs.
            
            The goal is not to maximize the number of categories.
            The goal is to determine whether important parts of
            the solution space remain unexplored.
            
            Return ONLY valid JSON in exactly this format:
            
            {{
                "continue_exploration": true,
                "covered_directions": [
                    "direction 1",
                    "direction 2",
                    "direction 3"
                ],
                "missing_directions": [
                    "missing direction 1",
                    "missing direction 2"
                ],
                "reason": "brief explanation"
            }}
            
            Set "continue_exploration" to false ONLY when the
            selected candidates cover the important solution
            directions sufficiently well.
            
            If important directions are missing, set it to true.
            
            Do not invent missing directions merely to force
            another exploration round.
    """

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=400,
            temperature=0.2,
            do_sample=True
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    del inputs
    del outputs
    torch.cuda.empty_cache()

    result = json.loads(response)

    state["missing_directions"] = result["missing_directions"]

    return result

In [21]:
#! Currently not in use
def converge(state):

    prompt = f"""
        Problem:
        {state["problem"]}
        
        Ideas:
        {state["ideas"]}
        
        Connections:
        {state["connections"]}
        
        Perspectives:
        {state["perspectives"]}
        
        attention_perturbations:
        {state["attention_perturbations"]}

        Cross Connections:
        {state["cross_connections"]}
        
        The exploration phase is complete.
        
        Now converge the reasoning.
        
        Identify the most promising ideas, connections, and cross-connections
        
        Consider:
        - relevance to the problem
        - potential usefulness
        - originality
        - feasibility
        - strength of the underlying reasoning
        
        Do not generate completely new ideas.
        Select and combine the strongest existing reasoning.
        
        Return ONLY valid JSON in exactly this format:
        
        {{
            "solutions": [
                "solution 1",
                "solution 2",
                "solution 3",
                "solution 4",
                "solution 5"
            ]
        }}
    """

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    # Cuda ops starts
    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.8,
            do_sample=True
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    del inputs
    del outputs
    torch.cuda.empty_cache()
    # cuda ops ends

    result = json.loads(response)

    return result["solutions"]

In [22]:
import json

def final_output(state, max_rounds=3):
    state["final_ideas"] = []

    for round_number in range(max_rounds):

        state["round"] = round_number + 1

        print(f"\n========== ROUND {state['round']} ==========")

        # Reset current-round exploration
        state["ideas"] = []
        state["connections"] = []
        state["perspectives"] = []
        state["attention_perturbations"] = []
        state["cross_connections"] = []
        state["selected_candidates"] = []
        
        # 1. Divergence
        diverge(state)

        # 2. Parallel exploration
        associative_search(state)
        perspective_switch(state)
        attention_perturb(state)

        # 3. Connect everything
        cross_connect(state)

        # 4. Laya selects promising candidates
        laya_select(state)
        
        state["final_ideas"].append({
            "round": state["round"],
            "ideas": state["selected_candidates"].copy()
        })

        # 5. Evaluate diversity / coverage
        evaluation = evaluate_coverage(state)

        print("\nEvaluation:")
        print(evaluation)

        # 6. If exploration is sufficient, stop
        if not evaluation["continue_exploration"]:
            print("\nExploration is sufficiently covered.")
            break

        # 7. Otherwise, use missing directions
        #    to guide the next divergence round
        print("\nImportant directions are still missing.")
        print("Missing directions:")
        print(state["missing_directions"])

    return state["selected_candidates"]

In [26]:
result = final_output(state)
print(result)


========== ROUND 1 ==========
Cross-connect: 53 originals + 25 new = 78 candidates for Laya
Laya decision making:

Candidate 1: P(true)=0.8398 | keep=True
Candidate 2: P(true)=0.7156 | keep=True
Candidate 3: P(true)=0.8516 | keep=True
Candidate 4: P(true)=0.8793 | keep=True
Candidate 5: P(true)=0.7667 | keep=True
Candidate 6: P(true)=0.8832 | keep=True
Candidate 7: P(true)=0.7807 | keep=True
Candidate 8: P(true)=0.6356 | keep=True
Candidate 9: P(true)=0.8023 | keep=True
Candidate 10: P(true)=0.7540 | keep=True
Candidate 11: P(true)=0.8363 | keep=True
Candidate 12: P(true)=0.6832 | keep=True
Candidate 13: P(true)=0.9020 | keep=True
Candidate 14: P(true)=0.7446 | keep=True
Candidate 15: P(true)=0.7860 | keep=True
Candidate 16: P(true)=0.8859 | keep=True
Candidate 17: P(true)=0.8268 | keep=True
Candidate 18: P(true)=0.8207 | keep=True
Candidate 19: P(true)=0.3082 | keep=False
Candidate 20: P(true)=0.4491 | keep=False
Candidate 21: P(true)=0.8530 | keep=True
Candidate 22: P(true)=0.5405 |

In [ ]:
print("IDEAS:")
for i, x in enumerate(state["ideas"], 1):
    print(f"{i}. {x}")


print("\nCONNECTIONS:")
for i, x in enumerate(state["connections"], 1):
    print(f"{i}. {x}")


print("\nPERTURBATIONS:")
for i, x in enumerate(state["attention_perturbations"], 1):
    print(f"{i}. {x}")


print("\nPERSPECTIVES:")
for i, x in enumerate(state["perspectives"], 1):
    print(f"{i}. {x}")


print("\nActive Memory:")
for i, x in enumerate(state["active_memory"], 1):
    print(f"{i}. {x}")

In [27]:
for i, x in enumerate(state["selected_candidates"], 1):
    print(f"{i}. {x}")

1. Launch a targeted Instagram giveaway with a unique prize to attract followers quickly, focusing on a specific niche related to your content.
2. Utilize Instagram Stories and Reels with consistent, high-quality content that includes trending hashtags and challenges to boost visibility and engagement.
3. Leverage User-Generated Content (UGC) Campaigns: Create a contest or challenge that encourages followers to share their own content related to your niche, which can then be featured on your Instagram feed, increasing visibility and engagement.
4. Collaborate with Micro-Influencers in Adjacent Niche Areas: Partner with influencers who have a smaller but highly engaged audience in related topics to cross-promote each other’s content, tapping into new follower bases and building credibility.
5. Implement a Story-based Engagement Strategy: Use Instagram Stories to run polls, Q&A sessions, or behind-the-scenes content that fosters direct interaction with your audience, building a loyal com